<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml06_regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

농어(Perch)의 무게 예측

*   농어의 모든 특성(길이, 대각선, 키, 두께)을 사용해서 무게 예측
    *   Weight ~ Length + Diagonal + Height + Width
*   KNN, Linear Regression 비교
*   특성들의 다차항을 포함하는 회귀
*   규제(Regularization)

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# Fish 데이터셋 준비

In [2]:
file_path = 'https://bit.ly/fish_csv_data'

In [3]:
fish = pd.read_csv(file_path)

In [4]:
fish.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    object 
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), object(1)
memory usage: 7.6+ KB


In [5]:
fish.head()

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [6]:
perch = fish[fish.Species == 'Perch']

In [7]:
perch.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, 72 to 127
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   56 non-null     object 
 1   Weight    56 non-null     float64
 2   Length    56 non-null     float64
 3   Diagonal  56 non-null     float64
 4   Height    56 non-null     float64
 5   Width     56 non-null     float64
dtypes: float64(5), object(1)
memory usage: 3.1+ KB


In [8]:
perch.head()

,Species,Weight,Length,Diagonal,Height,Width
72,Perch,5.9,8.4,8.8,2.1120,1.4080
73,Perch,32.0,13.7,14.7,3.5280,1.9992
74,Perch,40.0,15.0,16.0,3.8240,2.4320
75,Perch,51.5,16.2,17.2,4.5924,2.6316
76,Perch,70.0,17.4,18.5,4.5880,2.9415


In [10]:
perch.columns[2:]

Index(['Length', 'Diagonal', 'Height', 'Width'], dtype='object')

In [13]:
# 특성 배열(독립변수들) - 2차원 배열
x = perch[perch.columns[2:]].values
x.shape  #> (56, 4) = (n_samples, n_features)

(56, 4)

In [14]:
# 타겟 배열(종속변수) - 1차원 배열
y = perch.Weight.values
y.shape

(56,)

# 훈련 셋 vs 테스트 셋 나누기

In [16]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

In [17]:
x_train[:5]

array([[19.6   , 20.8   ,  5.1376,  3.0368],
       [22.    , 23.5   ,  5.875 ,  3.525 ],
       [18.7   , 19.4   ,  5.1992,  3.1234],
       [17.4   , 18.5   ,  4.588 ,  2.9415],
       [36.    , 38.3   , 10.6091,  6.7408]])

# 1차항들만 포함하는 회귀

## KNN

In [18]:
knn = KNeighborsRegressor()  # KNN 회귀 모델 생성

In [19]:
knn.fit(X=x_train, y=y_train)  # ML 모델 훈련

KNeighborsRegressor()

In [20]:
train_pred = knn.predict(X=x_train)  # 훈련 셋 (무게) 예측값
print(train_pred)

[  87.6  123.    79.6   70.6  723.   183.4  847.   847.  1020.   123.
   95.   123.   174.   248.  1043.   847.   174.   122.   248.   847.
  582.   224.   723.    60.   142.    60.   685.   694.2  248.   167.
  847.   122.   139.   123.  1020.   136.    79.6  685.   123.   193.
 1043.   659. ]


In [21]:
print(y_train)

[  85.  135.   78.   70.  700.  180.  850.  820. 1000.  120.   85.  130.
  225.  260. 1100.  900.  145.  115.  265. 1015.  514.  218.  685.   32.
  145.   40.  690.  840.  300.  170.  650.  110.  150.  110. 1000.  150.
   80.  700.  120.  197. 1100.  556.]


In [22]:
mean_squared_error(y_true=y_train, y_pred=train_pred)  # 훈련 셋 MSE

2986.5723809523806

In [23]:
knn.score(X=x_train, y=y_train)  # 훈련 셋 R2 score

0.97579760182756

In [24]:
test_pred = knn.predict(X=x_test)  # 테스트 셋 예측값

In [25]:
mean_squared_error(y_true=y_test, y_pred=test_pred)  # 테스트 셋 MSE

837.3100000000001

In [26]:
knn.score(X=x_test, y=y_test)  # 테스트 셋 R2 score

0.9916579819676246

KNN 회귀 모델은 테스트 셋에서 더 좋은 점수를 보여주고 있음 -> 과소적합(under-fitting)

## LinearRegression

*   Weight ~ Length + Diagonal + Height + Width
*   $ 무게 = w_0 + 길이 \times w_1 + 대각선 \times w_2 + 키 \times w_3 + 두께 \times w_4 $